# Tvorba aplikací pro generování obrázků (OpenAI)

Velké jazykové modely (LLM) nejsou jen pro generování textu. Můžete také generovat obrázky z textových popisů. Obrázky jako modality jsou užitečné v oblasti medicíny, architektury, turismu, vývoje her, marketingu a dalších. V této lekci pracujeme se současnými modely **GPT Image** na platformě OpenAI.

## Cíle učení

Na konci této lekce budete umět:

- Vysvětlit, co je generování obrázků a kde je užitečné.
- Pochopit rodinu modelů `gpt-image` a jak se liší od starších modelů DALL·E.
- Vytvořit aplikaci pro generování obrázků a upravovat obrázky.

## Co je generování obrázků?

Modely generování obrázků vytvářejí obrázky na základě textového zadání. Moderní modely jako `gpt-image` se během tréninku naučí vztah mezi textem a obrázky a pak postupně přeměňují náhodný šum na obrázek, který odpovídá vašemu zadání.

Dvě známé rodiny modelů pro obrázky jsou:

- **`gpt-image` (OpenAI)** — současná generace používaná v této lekci. Podporuje generování obrázků z textu i úpravy obrázků (inpainting s maskou).
- **Midjourney** — populární model třetí strany s vlastním servisem a workflow založeným na Discordu.

> Starší OpenAI modely obrázků — **DALL·E 2** a **DALL·E 3** — jsou již zastaralé. DALL·E 3 není pro nové nasazení dostupné, a funkce jako `create_variation` existovaly jen v DALL·E 2. Pro nové aplikace používejte modely `gpt-image`.

> **Důležité:** Modely `gpt-image` vracejí generovaný obrázek jako **base64** (`b64_json`), nikoli jako URL. Váš kód dekóduje base64 řetězec na bajty a uloží je — neexistuje URL obrázku ke stažení.


## Vytvoření vaší první aplikace pro generování obrázků

Co tedy potřebujete k vytvoření aplikace pro generování obrázků? Potřebujete následující knihovny:

- **python-dotenv**, důrazně doporučujeme používat tuto knihovnu pro uchování vašich tajných údajů v souboru *.env* mimo kód.
- **openai**, tuto knihovnu použijete pro komunikaci s OpenAI API.
- **pillow**, pro práci s obrázky v Pythonu.


1. Vytvořte soubor *.env* s následujícím obsahem:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Shromážděte výše uvedené knihovny do souboru s názvem *requirements.txt* takto:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Dále vytvořte virtuální prostředí a nainstalujte knihovny:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Pro Windows použijte následující příkazy k vytvoření a aktivaci vašeho virtuálního prostředí:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Přidejte následující kód do souboru s názvem *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Vytvořte objekt OpenAI (čte OPENAI_API_KEY z vašeho .env)
    client = openai.OpenAI()


    try:
        # Vytvořte obrázek pomocí API pro generování obrázků
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Sem zadejte text výzvy
            size='1024x1024',
            n=1
        )
        # Nastavte adresář pro uložený obrázek
        image_dir = os.path.join(os.curdir, 'images')

        # Pokud adresář neexistuje, vytvořte ho
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicializujte cestu k obrázku (pozor, typ souboru by měl být png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Modely gpt-image vrací obrázek jako base64 (b64_json), nikoli URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Zobrazte obrázek ve výchozím prohlížeči obrázků
        image = Image.open(image_path)
        image.show()

    # zachytit výjimky
    except openai.BadRequestError as err:
        print(err)

    ```

Vysvětlíme tento kód:

- Nejprve importujeme potřebné knihovny, včetně knihovny OpenAI, knihovny dotenv, modulu base64 a knihovny Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Poté vytvoříme klienta, který načte API klíč z vašeho souboru ``.env``.

    ```python
    # Vytvořit objekt OpenAI
    client = openai.OpenAI()
    ```

- Dále vygenerujeme obrázek:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Zadejte zde text svého požadavku
        size='1024x1024',
        n=1
    )
    ```

    Modely `gpt-image` vracejí obrázek jako **base64** řetězec v `data[0].b64_json`. Dekódujeme jej na bajty a zapíšeme do souboru — není zde žádná URL ke stažení.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Nakonec otevřeme obrázek a použijeme standardní prohlížeč obrázků k jeho zobrazení:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Více informací o generování obrázku

Podívejme se na parametry funkce `images.generate`:

- **model**, je model obrázku, který se má použít (například `gpt-image-1`).
- **prompt**, je textový prompt použitý k vytvoření obrázku. Zde je to "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, je velikost generovaného obrázku (například `1024x1024`, `1536x1024`, `1024x1536` nebo `"auto"`).
- **n**, je počet generovaných obrázků. Zde generujeme jeden.

> Modely obrázků nepřijímají parametr `temperature` — to je řídící parametr generování textu. Pro větší rozmanitost znovu zavolejte API; pro menší rozmanitost udělejte váš prompt specifickější.

## Další možnosti generování obrázků

Viděli jste, jak vygenerovat obrázek několika řádky Pythonu. Modely `gpt-image` také dokáží **upravit** stávající obrázek. Poskytnutím obrázku, volitelné **masky** (která označuje oblast k úpravě) a promptu můžete změnit část obrázku — například přidat králíkovi klobouk.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# úpravy jsou také vráceny jako base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Základní obrázek obsahuje pouze králíka; výsledný obrázek má na králíkovi klobouk.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
